# 04. 위험격자 생성 — 100m 격자 + 도달거리

## 이 노트북이 하는 일
초량·좌천 경계 안을 **100m 격자**로 나누고, 각 격자를 도로망에 스냅해 **최근접 119안전센터까지의 도로거리**(도달지연 요소)를 계산한다. 인구(65+·독거)는 05에서 조인.

## 왜 이렇게 설계했나 (설계 이유)
- **왜 100m 격자인가:** 분석·최적화(MCLP)의 기본 단위. 산복도로 골목 수준을 담되 너무 잘게 쪼개지 않는 절충.
- **왜 경계 내부만(중심점 기준)인가:** 대상지 밖 격자는 해석 대상이 아님. 격자 '중심점이 경계 안'일 때만 채택.
- **왜 도달지연을 여기서 거리로 먼저 넣나:** 위험 3요소 중 '도달지연'은 인구와 무관하게 그래프만으로 계산 가능. 인구는 05에서 곱함.
- **왜 안전센터→전노드 Dijkstra인가:** 각 격자에서 센터까지 일일이 계산하면 느림. 센터에서 한 번 퍼뜨리면(single-source) 모든 노드 거리가 한 번에 나온다.
- **왜 인구 칸을 비워두나:** SGIS 인구 데이터 조인은 05의 일. 여기선 스키마(빈 칸)만 만들어 다음 단계가 채우게 함.

## 데이터 출처
- 경계: OSM 행정경계 / 그래프: 03(M2, OSM+SRTM). 인구: 미조인(05에서 SGIS).

In [ ]:
import os, warnings                       # 폴더·경고
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd            # 수치·표
import geopandas as gpd                     # 지리 표(격자·경계)
import networkx as nx                       # Dijkstra(도달거리)
import osmnx as ox                          # 그래프 로드·스냅
import folium                               # 도달거리 지도
from shapely.geometry import box, Point     # box: 격자 사각형 / Point: 격자 중심점
from shapely.ops import unary_union         # 여러 동 경계 합치기
print("osmnx", ox.__version__)
CRS_WGS, CRS_M = 4326, 5186                 # 위경도 / 평면(m)
GRID = 100                                  # 격자 한 변 100m
os.makedirs("outputs", exist_ok=True)

## 1. 경계(초량·좌천) + M2 그래프 로드

In [ ]:
DONGGU_BBOX = box(129.020, 35.100, 129.075, 35.155)                                  # 동구 조회 사각형
boundary_geom = None
try:
    adm = ox.features_from_polygon(DONGGU_BBOX, tags={"boundary":"administrative"})   # 행정경계 조회
    adm = adm[adm.geometry.geom_type.isin(["Polygon","MultiPolygon"])].copy()         # 면만
    nm = adm["name"].astype(str) if "name" in adm.columns else pd.Series([""]*len(adm), index=adm.index)
    sel = adm[nm.str.contains("초량|좌천", na=False)]                                  # 초량·좌천만
    if len(sel) > 0:
        boundary_geom = unary_union(sel.geometry)                                     # 합치기
        print("경계:", ", ".join(sorted(set(nm[sel.index]))))
except Exception as e:
    print("경계 조회 실패:", e)
if boundary_geom is None:                                                             # 실패 시 fallback
    boundary_geom = box(129.030, 35.110, 129.060, 35.145)
    print("fallback bbox")
boundary_m = gpd.GeoSeries([boundary_geom], crs=CRS_WGS).to_crs(CRS_M).iloc[0]         # 격자 생성은 미터 좌표계에서

def _sf(x):                                                                           # M2 저장 시 빈 문자열("")로 둔 grade 등을
    try: return float(x)                                                              # 안전하게 실수로 변환(빈 값은 NaN)
    except: return float("nan")
G = ox.load_graphml("outputs/graph_drive_M2.graphml",                                 # 03이 저장한 M2 그래프 로드
    edge_dtypes={"grade":_sf,"grade_abs":_sf,"elev_u":_sf,"elev_v":_sf,"width_est":_sf,"length":float},
    node_dtypes={"elev":_sf})
station_nodes = [n for n,d in G.nodes(data=True) if d.get("node_type")=="station"]     # 안전센터로 스냅된 노드들
hosp_nodes    = [n for n,d in G.nodes(data=True) if d.get("node_type")=="hospital"]    # 병원 노드들
print("그래프 노드:", G.number_of_nodes(), "| 안전센터 노드:", len(station_nodes), "| 병원 노드:", len(hosp_nodes))

## 2. 100m 격자 생성 (경계 내부만, EPSG:5186)

In [ ]:
minx, miny, maxx, maxy = boundary_m.bounds                                            # 경계의 바깥 사각형 좌표
xs = np.arange(np.floor(minx/GRID)*GRID, maxx, GRID)                                  # 100m 간격 x 좌표들
ys = np.arange(np.floor(miny/GRID)*GRID, maxy, GRID)                                  # 100m 간격 y 좌표들
cells, cxy = [], []                                                                   # 격자 폴리곤 / 중심점 저장
for x in xs:                                                                          # 격자 격자무늬 순회
    for y in ys:
        c = box(x, y, x+GRID, y+GRID)                                                # 한 칸(100x100m) 사각형
        if c.centroid.within(boundary_m):                                            # 중심점이 경계 안일 때만 채택
            cells.append(c); cxy.append((x+GRID/2, y+GRID/2))                         # 폴리곤·중심좌표 저장
grid = gpd.GeoDataFrame({"grid_id":[f"G{i:04d}" for i in range(len(cells))]},         # 격자 표(고유 id 부여)
                        geometry=cells, crs=CRS_M)
grid["cx"] = [p[0] for p in cxy]; grid["cy"] = [p[1] for p in cxy]                    # 중심 x/y 컬럼
print("격자 수:", len(grid))

## 3. 격자 → 최근접 도로노드 스냅 + 최근접 안전센터 거리
도달지연 요소. 거리(length) 기준 임시.

In [ ]:
grid["entry_node"] = ox.distance.nearest_nodes(G, X=grid["cx"].values, Y=grid["cy"].values)  # 각 격자중심의 최근접 도로노드(진입점)

best = {}                                                                             # 노드별 '가장 가까운 센터까지 거리'
for s in station_nodes:                                                               # 각 안전센터에서
    for n, d in nx.single_source_dijkstra_path_length(G, s, weight="length").items(): # 그 센터→모든 노드 최단거리 한 번에
        if n not in best or d < best[n]:                                              # 여러 센터 중 최소값만 유지
            best[n] = d
grid["dist_station_m"] = grid["entry_node"].map(lambda n: best.get(n, np.nan))         # 격자 진입점의 센터거리 상속
print("도달거리 산출된 격자:", grid["dist_station_m"].notna().sum(), "/", len(grid))
print("최근접 안전센터 거리 min/median/max: %.0f / %.0f / %.0f m" % (
    grid["dist_station_m"].min(), grid["dist_station_m"].median(), grid["dist_station_m"].max()))

## 4. 위험 스키마 (인구 자리 확보)
pop_75, solo_ratio는 05에서 SGIS 조인 시 채움. risk도 그때 계산.

In [ ]:
grid["pop_total"]  = np.nan   # 총인구 (05에서 SGIS)
grid["pop_75"]     = np.nan   # 75+/65+ 인구 = 발생확률 (05)
grid["solo_ratio"] = np.nan   # 독거 프록시 = 목격불가 (05)
grid["reach_proxy"] = grid["dist_station_m"]          # 도달지연 프록시(거리) — 이미 계산됨
grid["in_scope"]   = np.nan   # 대상지 여부(도달시간>4분) — 도달시간 산정 후
grid["risk"]       = np.nan   # 위험도 = 발생확률 × 목격불가 × 도달지연 (05)

cols = ["grid_id","cx","cy","entry_node","dist_station_m","reach_proxy",             # 저장할 컬럼 순서 정리
        "pop_total","pop_75","solo_ratio","in_scope","risk","geometry"]
grid = grid[cols]
grid.to_parquet("outputs/grid_donggu_A1.parquet")                                     # 격자 뼈대 저장(05가 읽음)
try: grid.to_file("outputs/grid_donggu_A1.gpkg", driver="GPKG")                       # QGIS용
except Exception as e: print("gpkg 경고:", e)
print("저장: outputs/grid_donggu_A1.parquet / .gpkg")
grid.head()

## 5. 시각화 — 격자별 최근접 안전센터 거리

In [ ]:
gw = grid.to_crs(CRS_WGS)                                                             # 지도용 위경도 변환
d = gw["dist_station_m"]
dmax = np.nanpercentile(d, 95) if d.notna().any() else 1.0                            # 색 정규화 상한(상위 5% 절단)
def col(v):                                                                           # 거리→색 (초록=가까움, 빨강=멂)
    if pd.isna(v): return "#cccccc"
    t = min(v/dmax, 1.0)
    r = int(26+(215-26)*t); g = int(150+(25-150)*t); b = int(65+(28-65)*t)
    return f"#{r:02x}{g:02x}{b:02x}"

m = folium.Map(location=[35.128, 129.045], zoom_start=14, tiles="cartodbpositron")
folium.GeoJson(gpd.GeoSeries([boundary_geom], crs=CRS_WGS).__geo_interface__,          # 경계선
               name="경계", style_function=lambda x:{"color":"#2c7fb8","weight":2,"fill":False}).add_to(m)
for _, r in gw.iterrows():                                                            # 격자마다 색칠
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=lambda x, c=col(r["dist_station_m"]): {"color":c,"weight":0.3,"fillColor":c,"fillOpacity":0.55},
        tooltip=f'{r["grid_id"]}: {"" if pd.isna(r["dist_station_m"]) else int(r["dist_station_m"])}m').add_to(m)
st_pts = gpd.GeoSeries([Point(G.nodes[n]["x"], G.nodes[n]["y"]) for n in station_nodes],  # 안전센터 위치(파랑 점)
                       crs=CRS_M).to_crs(CRS_WGS)
for p in st_pts:
    folium.CircleMarker([p.y, p.x], radius=4, color="blue", fill=True, fillOpacity=0.9,
                        tooltip="119안전센터").add_to(m)
folium.LayerControl().add_to(m)
m.save("outputs/grid_reach_A1.html")                                                  # 도달거리 지도 저장
print("지도 저장: outputs/grid_reach_A1.html")
m